# 🚀 RAG local en GPU: pregúntale al curso, y verifica afirmaciones con Wikipedia
*Curso de IA Generativa · 🚀 Llévalo a Colab · [MiAtlas.cl](https://miatlas.cl) · Licencia [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/deed.es)*

**Objetivo:** construir un sistema RAG (*Retrieval-Augmented Generation*) **sin claves pagadas**: busca los fragmentos
relevantes y le pide a un modelo abierto que responda **solo con ellos, citando la fuente**.

- **Parte 1 · Pregúntale al curso:** el corpus son los cuadernos del curso «Google Colab» de MiAtlas.cl (CC BY 4.0).
- **Parte 2 · Verificar afirmaciones:** afirmaciones de **FEVER** (CC BY-SA 3.0) contrastadas con Wikipedia en vivo.

In [ ]:
# Preparación: ¿Colab?, ¿GPU? y modo de prueba
import os, gc, time
try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False
PRUEBA_RAPIDA = os.environ.get("MIATLAS_PRUEBA") == "1"   # solo para la prueba automática de MiAtlas.cl

import torch
DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"
print("Colab:", EN_COLAB, "| dispositivo:", DISPOSITIVO, "|", torch.cuda.get_device_name(0) if DISPOSITIVO == "cuda" else "")
if DISPOSITIVO == "cpu":
    print("⚠ Sin GPU. En Colab: Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU.")

def liberar():
    """Devuelve la memoria de la GPU después de borrar un modelo («del modelo»)."""
    gc.collect()
    if DISPOSITIVO == "cuda":
        torch.cuda.empty_cache()

In [ ]:
%pip install -q sentence-transformers faiss-cpu transformers accelerate bitsandbytes datasets langchain-community wikipedia

## Parte 1 · Pregúntale al curso

### 1. El corpus: los cuadernos del curso, desde GitHub

In [ ]:
import json, re, urllib.request

REPO = "psalgadm7/miatlas-cuadernos"
with urllib.request.urlopen(f"https://api.github.com/repos/{REPO}/contents/cuadernos") as r:
    archivos = [a["name"] for a in json.load(r) if re.match(r"^\d\d-.*\.ipynb$", a["name"])]
documentos = []
for nombre in archivos:
    with urllib.request.urlopen(f"https://raw.githubusercontent.com/{REPO}/main/cuadernos/{nombre}") as r:
        nb = json.load(r)
    texto = "\n\n".join("".join(c["source"]) for c in nb["cells"])
    documentos.append({"fuente": nombre, "texto": texto})
print(len(documentos), "cuadernos ·", sum(len(d["texto"]) for d in documentos), "caracteres")

### 2. Fragmentar

Un cuaderno entero no cabe bien en una búsqueda: se parte en **fragmentos** de unos 800 caracteres que se traslapan
un poco, para no cortar una idea por la mitad.

In [ ]:
def fragmentar(texto, largo=800, traslape=150):
    partes, inicio = [], 0
    while inicio < len(texto):
        partes.append(texto[inicio:inicio + largo])
        inicio += largo - traslape
    return partes

fragmentos = [{"fuente": d["fuente"], "texto": f} for d in documentos for f in fragmentar(d["texto"])]
print(len(fragmentos), "fragmentos. Ejemplo:\n", fragmentos[5]["texto"][:300])

### 3. Embeddings y un índice FAISS

`paraphrase-multilingual-MiniLM-L12-v2` (Apache 2.0) convierte cada fragmento en un vector de 384 números: textos de
significado parecido quedan cerca, aunque usen otras palabras. **FAISS** busca los más cercanos entre miles en
milisegundos.

In [ ]:
import numpy as np, faiss
from sentence_transformers import SentenceTransformer

vectorizador = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", device=DISPOSITIVO)
inicio = time.perf_counter()
vectores = vectorizador.encode([f["texto"] for f in fragmentos], batch_size=64, normalize_embeddings=True)
print(f"{len(vectores)} vectores de {vectores.shape[1]} dimensiones en {time.perf_counter() - inicio:.1f} s ({DISPOSITIVO})")
indice = faiss.IndexFlatIP(vectores.shape[1])        # producto interno = coseno, porque están normalizados
indice.add(vectores.astype("float32"))

def buscar(pregunta, k=4):
    q = vectorizador.encode([pregunta], normalize_embeddings=True).astype("float32")
    puntajes, posiciones = indice.search(q, k)
    return [{**fragmentos[i], "puntaje": float(p)} for p, i in zip(puntajes[0], posiciones[0])]

for r in buscar("¿Cómo guardo una clave sin escribirla en el cuaderno?"):
    print(f"{r['puntaje']:.2f} · {r['fuente']}")

### 4. ¿Recupera lo correcto? Medirlo antes de generar

Si la búsqueda falla, el modelo responderá mal aunque sea bueno. Se mide con preguntas cuya fuente conocemos.

In [ ]:
pruebas = [
    ("¿Cómo guardo una clave sin escribirla en el cuaderno?", "04-secretos.ipynb"),
    ("¿Cómo conecto mi Google Drive?", "02-archivos-y-drive.ipynb"),
    ("¿Cuántas horas puede correr un cuaderno en el plan gratis?", "01-entorno.ipynb"),
    ("¿Cómo mido bien el tiempo de un cálculo en la GPU?", "06-primera-gpu.ipynb"),
    ("¿Cómo retomo un entrenamiento después de una desconexión?", "09-buenas-practicas.ipynb"),
]
aciertos = sum(any(r["fuente"] == esperada for r in buscar(p, k=4)) for p, esperada in pruebas)
print(f"Recuperación: la fuente correcta aparece entre las 4 primeras en {aciertos} de {len(pruebas)} preguntas")

### 5. Generar la respuesta con un modelo abierto

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODELO = "Qwen/Qwen2.5-1.5B-Instruct"        # con GPU puedes probar "Qwen/Qwen2.5-7B-Instruct" en 4 bits
opciones = {"device_map": "auto", "torch_dtype": torch.float16} if DISPOSITIVO == "cuda" else {}
if "7B" in MODELO:
    opciones = {"device_map": "auto", "quantization_config": BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)}
tok = AutoTokenizer.from_pretrained(MODELO)
llm = AutoModelForCausalLM.from_pretrained(MODELO, **opciones)

def generar(instruccion, tokens=300):
    entrada = tok.apply_chat_template([{"role": "user", "content": instruccion}], add_generation_prompt=True,
                                      return_tensors="pt").to(llm.device)
    salida = llm.generate(entrada, max_new_tokens=tokens, do_sample=False, pad_token_id=tok.eos_token_id)
    return tok.decode(salida[0, entrada.shape[1]:], skip_special_tokens=True).strip()

def responder(pregunta, k=4):
    contexto = "\n\n".join(f"[{r['fuente']}]\n{r['texto']}" for r in buscar(pregunta, k))
    return generar(f"""Responde la pregunta usando SOLO el contexto. Si el contexto no alcanza, di «No lo encuentro en el curso».
Al final, indica entre corchetes los archivos que usaste.

CONTEXTO:
{contexto}

PREGUNTA: {pregunta}""")

print(responder("¿Cómo guardo una clave sin escribirla en el cuaderno?"))
print("\n---\n", responder("¿Cuál es la capital de Australia?"))     # fuera del curso: debería decir que no lo encuentra

## Parte 2 · Verificar afirmaciones con Wikipedia (como en FEVER)

FEVER reúne afirmaciones escritas a partir de Wikipedia, cada una marcada como **SUPPORTS** (la evidencia la apoya),
**REFUTES** (la contradice) o **NOT ENOUGH INFO**. Aquí el sistema busca en Wikipedia en vivo y el modelo decide.

In [ ]:
from datasets import load_dataset

fever = load_dataset("copenlu/fever_gold_evidence", split="validation").to_pandas()
muestra = fever.groupby("label").sample(n=2 if PRUEBA_RAPIDA else 10, random_state=0).reset_index(drop=True)
muestra[["claim", "label"]]

In [ ]:
from langchain_community.retrievers import WikipediaRetriever

wikipedia = WikipediaRetriever(top_k_results=2, doc_content_chars_max=1500, lang="en")
ETIQUETAS = ["SUPPORTS", "REFUTES", "NOT ENOUGH INFO"]

def verificar(afirmacion):
    evidencia = "\n\n".join(d.page_content for d in wikipedia.invoke(afirmacion))
    texto = generar(f"""Evidence from Wikipedia:
{evidencia}

Claim: {afirmacion}
Does the evidence SUPPORT the claim, REFUTE it, or is there NOT ENOUGH INFO? Answer with exactly one of:
SUPPORTS, REFUTES, NOT ENOUGH INFO.""", tokens=10).upper()
    return next((e for e in ETIQUETAS if e in texto), "NOT ENOUGH INFO")

muestra["prediccion"] = [verificar(a) for a in muestra["claim"]]
print(f"Exactitud: {(muestra['prediccion'] == muestra['label']).mean():.0%}")
muestra[["claim", "label", "prediccion"]]

> **¿El error es de la búsqueda o del modelo?** FEVER trae la **evidencia de oro** (la oración exacta de Wikipedia).
> Repite la verificación dándole esa evidencia en vez de la búsqueda: si acierta mucho más, el cuello de botella es
> la recuperación.

## Pruébalo tú

Mide la recuperación de la Parte 1 con `k=1`, `k=2` y `k=8`, y guarda en `mejor_k` el menor `k` que encuentra la fuente
correcta en las 5 preguntas.

In [ ]:
mejor_k = None   # ← tu código

In [ ]:
# Comprobación
if mejor_k is None:
    print("Aún no lo resuelves.")
elif 1 <= mejor_k <= 8:
    print("✅ Registrado: ¿qué cuesta usar un k más grande?")
else:
    print("❌ Revisa tu respuesta")

## Resumen

- RAG = **buscar** (embeddings + índice) y **generar** con el contexto encontrado, citando la fuente.
- Mide la recuperación por separado: un buen modelo no arregla una mala búsqueda.
- Todo esto corre gratis en Colab, sin claves pagadas.

**Fuentes:** [sentence-transformers](https://sbert.net/) · [paraphrase-multilingual-MiniLM-L12-v2](https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2) · [FAISS](https://faiss.ai/) · [LangChain · WikipediaRetriever](https://python.langchain.com/docs/integrations/retrievers/wikipedia/) · [FEVER (Thorne et al., 2018)](https://fever.ai/dataset/fever.html) · [copenlu/fever_gold_evidence](https://huggingface.co/datasets/copenlu/fever_gold_evidence)